# Lab 04 — SCD Type 1 and Type 2 Comparison

This notebook compares the two product dimensions created in the previous notebooks. It does not write or mutate either table. Instead, it validates their different history semantics, compares business-key coverage, demonstrates current-state and point-in-time query patterns, and records evidence for the Lab 4 README.

## Learning objectives

- prove that SCD Type 1 keeps only the latest product state;
- prove that SCD Type 2 preserves previous versions with valid time boundaries;
- confirm that both dimensions cover the same product business keys;
- compare storage, query, audit, and replay trade-offs;
- select the correct pattern for an analytics requirement.

## 1. Load the shared configuration

The configuration notebook supplies catalog, schema, table names, validation flags, and common Lab 4 parameters. This comparison expects `lab04_05_scd_type1.ipynb` and `lab04_06_scd_type2.ipynb` to have completed successfully.

In [0]:
%run ./lab04_00_config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

product_scd1_table = table_names["product_scd1"]
product_scd2_table = table_names["product_scd2"]

print(f"SCD Type 1 table: {product_scd1_table}")
print(f"SCD Type 2 table: {product_scd2_table}")
print(f"Validation enabled: {run_validation}")

## 2. Verify prerequisites

Both targets must already exist and contain data. Failing early here prevents misleading comparisons against an empty or missing dimension.

In [0]:
missing_tables = [
    table_name
    for table_name in [product_scd1_table, product_scd2_table]
    if not spark.catalog.tableExists(table_name)
]
if missing_tables:
    raise ValueError(
        "Required SCD tables are missing: " + ", ".join(missing_tables) + ". "
        "Run lab04_05_scd_type1 and lab04_06_scd_type2 first."
    )

scd1_df = spark.table(product_scd1_table)
scd2_df = spark.table(product_scd2_table)
scd1_row_count = scd1_df.count()
scd2_row_count = scd2_df.count()

if scd1_row_count == 0 or scd2_row_count == 0:
    raise ValueError(
        f"SCD targets must be populated: scd1={scd1_row_count}, scd2={scd2_row_count}."
    )

print(f"✅ Prerequisites passed: SCD1={scd1_row_count:,} rows; SCD2={scd2_row_count:,} rows.")

## 3. Conceptual comparison

Type 1 answers **what is true now?** Type 2 additionally answers **what was true at a historical point in time?** The patterns deliberately optimize for different business needs.

In [0]:
comparison_matrix_df = spark.createDataFrame(
    [
        ("History", "Overwrites old attributes", "Adds a new version and closes the old one"),
        ("Rows per business key", "Exactly one", "One current plus zero or more historical rows"),
        ("Primary analytical use", "Current-state reporting", "Historical and point-in-time reporting"),
        ("Change operation", "Matched UPDATE", "Close current row, then INSERT new version"),
        ("Storage cost", "Lower", "Higher because history is retained"),
        ("Auditability", "Limited", "Strong business-history trace"),
        ("Typical example", "Correct a product description", "Track price or customer-segment history"),
    ],
    ["criterion", "scd_type_1", "scd_type_2"],
)
display(comparison_matrix_df)

## 4. Calculate dimension metrics

The metrics make the structural difference visible. SCD1 should have the same row count and distinct business-key count. SCD2 should have one current row per business key plus additional historical rows.

In [0]:
scd1_metrics = (
    scd1_df
    .agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("stock_code").alias("business_keys"),
        F.sum(F.col("is_current").cast("long")).alias("current_rows"),
        F.sum((~F.col("is_current")).cast("long")).alias("historical_rows"),
    )
    .first()
    .asDict()
)
scd2_metrics = (
    scd2_df
    .agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("stock_code").alias("business_keys"),
        F.sum(F.col("is_current").cast("long")).alias("current_rows"),
        F.sum((~F.col("is_current")).cast("long")).alias("historical_rows"),
    )
    .first()
    .asDict()
)

dimension_metrics_df = spark.createDataFrame(
    [
        ("SCD Type 1", *[int(scd1_metrics[name] or 0) for name in ["total_rows", "business_keys", "current_rows", "historical_rows"]]),
        ("SCD Type 2", *[int(scd2_metrics[name] or 0) for name in ["total_rows", "business_keys", "current_rows", "historical_rows"]]),
    ],
    ["dimension_type", "total_rows", "business_keys", "current_rows", "historical_rows"],
)
display(dimension_metrics_df)

## 5. Validate SCD Type 1 semantics

A valid Type 1 dimension has one row per `stock_code`, all rows are current, and no row has an end timestamp. Old business values are intentionally not retained in the table.

In [0]:
scd1_duplicate_keys = (
    scd1_df.groupBy("stock_code").count().filter(F.col("count") != 1).count()
)
scd1_non_current_rows = scd1_df.filter(~F.col("is_current")).count()
scd1_closed_rows = scd1_df.filter(F.col("effective_to").isNotNull()).count()
scd1_null_keys = scd1_df.filter(
    F.col("stock_code").isNull() | F.col("product_sk").isNull()
).count()

scd1_checks = {
    "duplicate_business_keys": scd1_duplicate_keys,
    "non_current_rows": scd1_non_current_rows,
    "closed_rows": scd1_closed_rows,
    "null_business_or_surrogate_keys": scd1_null_keys,
}
if run_validation and any(value != 0 for value in scd1_checks.values()):
    raise AssertionError(f"SCD Type 1 comparison checks failed: {scd1_checks}")

display(spark.createDataFrame(list(scd1_checks.items()), ["quality_rule", "violation_count"]))
print("✅ SCD Type 1 contains exactly one current-state row per product.")

## 6. Validate SCD Type 2 semantics

A valid Type 2 dimension has exactly one current version per business key, unique version numbers, open-ended current rows, closed historical rows, and contiguous effective-time boundaries.

In [0]:
scd2_current_counts_df = (
    scd2_df.select("stock_code").distinct()
    .join(
        scd2_df.filter(F.col("is_current"))
        .groupBy("stock_code")
        .count()
        .withColumnRenamed("count", "current_count"),
        "stock_code",
        "left",
    )
    .fillna({"current_count": 0})
)
scd2_current_key_violations = (
    scd2_current_counts_df.filter(F.col("current_count") != 1).count()
)
scd2_duplicate_versions = (
    scd2_df.groupBy("stock_code", "version_number")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
scd2_invalid_current_dates = scd2_df.filter(
    F.col("is_current") & F.col("effective_to").isNotNull()
).count()
scd2_invalid_history_dates = scd2_df.filter(
    (~F.col("is_current"))
    & (F.col("effective_to").isNull() | (F.col("effective_to") <= F.col("effective_from")))
).count()

version_window = Window.partitionBy("stock_code").orderBy("effective_from", "version_number")
scd2_boundary_df = (
    scd2_df
    .withColumn("next_effective_from", F.lead("effective_from").over(version_window))
    .withColumn(
        "invalid_boundary",
        F.when(
            F.col("next_effective_from").isNotNull(),
            F.col("effective_to").isNull()
            | (F.col("effective_to") != F.col("next_effective_from")),
        ).otherwise(F.col("effective_to").isNotNull()),
    )
)
scd2_invalid_boundaries = scd2_boundary_df.filter(F.col("invalid_boundary")).count()

scd2_checks = {
    "current_key_violations": scd2_current_key_violations,
    "duplicate_version_numbers": scd2_duplicate_versions,
    "invalid_current_end_dates": scd2_invalid_current_dates,
    "invalid_historical_end_dates": scd2_invalid_history_dates,
    "invalid_temporal_boundaries": scd2_invalid_boundaries,
}
if run_validation and any(value != 0 for value in scd2_checks.values()):
    raise AssertionError(f"SCD Type 2 comparison checks failed: {scd2_checks}")

display(spark.createDataFrame(list(scd2_checks.items()), ["quality_rule", "violation_count"]))
print("✅ SCD Type 2 has one current row per product and valid historical boundaries.")

## 7. Compare business-key coverage

Both dimensions were derived from the same Silver transaction table, so they must cover the same set of `stock_code` values. Their row counts differ only because Type 2 retains history.

In [0]:
scd1_keys_df = scd1_df.select("stock_code").distinct()
scd2_keys_df = scd2_df.select("stock_code").distinct()
missing_from_scd2_df = scd1_keys_df.join(scd2_keys_df, "stock_code", "left_anti")
missing_from_scd1_df = scd2_keys_df.join(scd1_keys_df, "stock_code", "left_anti")
missing_from_scd2 = missing_from_scd2_df.count()
missing_from_scd1 = missing_from_scd1_df.count()

if run_validation and (missing_from_scd2 != 0 or missing_from_scd1 != 0):
    raise AssertionError(
        f"Business-key coverage differs: missing from SCD2={missing_from_scd2}, "
        f"missing from SCD1={missing_from_scd1}."
    )

coverage_df = spark.createDataFrame(
    [
        ("missing_from_scd2", missing_from_scd2),
        ("missing_from_scd1", missing_from_scd1),
    ],
    ["coverage_check", "missing_keys"],
)
display(coverage_df)
print("✅ SCD Type 1 and Type 2 cover the same product business keys.")

## 8. Inspect changed products side by side

The previous notebooks intentionally applied different controlled descriptions and price changes to make each pattern independently visible. Therefore, current attribute values are displayed for comparison but are not expected to match. The important distinction is that Type 1 has only the latest row while Type 2 also exposes earlier versions.

In [0]:
versioned_keys_df = (
    scd2_df.groupBy("stock_code")
    .agg(
        F.count("*").alias("scd2_version_count"),
        F.max("version_number").alias("latest_version_number"),
    )
    .filter(F.col("scd2_version_count") > 1)
)

current_comparison_df = (
    versioned_keys_df.alias("keys")
    .join(
        scd1_df.select(
            "stock_code",
            F.col("description").alias("scd1_current_description"),
            F.col("latest_observed_price").alias("scd1_current_price"),
        ).alias("scd1"),
        "stock_code",
    )
    .join(
        scd2_df.filter(F.col("is_current")).select(
            "stock_code",
            F.col("description").alias("scd2_current_description"),
            F.col("latest_observed_price").alias("scd2_current_price"),
            F.col("effective_from").alias("scd2_current_from"),
        ).alias("scd2"),
        "stock_code",
    )
    .select(
        "stock_code", "scd2_version_count", "latest_version_number",
        "scd1_current_description", "scd1_current_price",
        "scd2_current_description", "scd2_current_price", "scd2_current_from",
    )
    .orderBy("stock_code")
)
display(current_comparison_df)

scd2_history_evidence_df = (
    scd2_df.join(versioned_keys_df.select("stock_code"), "stock_code", "inner")
    .select(
        "stock_code", "version_number", "description", "latest_observed_price",
        "effective_from", "effective_to", "is_current", "source_batch_id",
    )
    .orderBy("stock_code", "version_number")
)
display(scd2_history_evidence_df)

## 9. Demonstrate analytics query patterns

SCD1 is naturally queried as a current-state table. SCD2 current reporting filters `is_current=true`; historical reporting supplies an as-of timestamp between `effective_from` and `effective_to`.

In [0]:
scd1_current_view_df = scd1_df.select(
    "product_sk", "stock_code", "description", "latest_observed_price"
)
scd2_current_view_df = (
    scd2_df.filter(F.col("is_current"))
    .select(
        "product_version_sk", "product_sk", "stock_code",
        "description", "latest_observed_price", "version_number", "effective_from",
    )
)

as_of_timestamp = (
    scd2_df.filter(~F.col("is_current"))
    .agg(F.min("effective_from").alias("as_of_timestamp"))
    .first()["as_of_timestamp"]
)
if as_of_timestamp is None:
    as_of_timestamp = scd2_df.agg(F.max("effective_from")).first()[0]

scd2_as_of_df = (
    scd2_df
    .filter(
        (F.col("effective_from") <= F.lit(as_of_timestamp))
        & (F.col("effective_to").isNull() | (F.col("effective_to") > F.lit(as_of_timestamp)))
    )
    .select(
        "stock_code", "description", "latest_observed_price",
        "version_number", "effective_from", "effective_to",
    )
)

print(f"SCD1 current rows: {scd1_current_view_df.count():,}")
print(f"SCD2 current rows: {scd2_current_view_df.count():,}")
print(f"SCD2 point-in-time example: {as_of_timestamp}")
display(scd2_as_of_df.orderBy("stock_code").limit(30))

## 10. Review Delta history

Delta transaction history is separate from SCD business history. Delta history records physical table operations; the Type 2 rows record business-valid attribute history that analysts can query directly.

In [0]:
scd1_delta_history_df = (
    spark.sql(f"DESCRIBE HISTORY {product_scd1_table}")
    .select("version", "timestamp", "operation", "operationMetrics")
    .withColumn("dimension_type", F.lit("SCD Type 1"))
)
scd2_delta_history_df = (
    spark.sql(f"DESCRIBE HISTORY {product_scd2_table}")
    .select("version", "timestamp", "operation", "operationMetrics")
    .withColumn("dimension_type", F.lit("SCD Type 2"))
)
display(
    scd1_delta_history_df.unionByName(scd2_delta_history_df)
    .orderBy(F.col("timestamp").desc())
    .limit(20)
)

## 11. Final validation summary

The final assertions connect the implementation to the lab requirements. Type 1 must contain no historical rows. Type 2 must retain history while preserving exactly one current row for every product. Running this notebook repeatedly is safe because it performs reads and assertions only.

In [0]:
final_checks = [
    ("scd1_one_row_per_business_key", scd1_metrics["total_rows"] == scd1_metrics["business_keys"]),
    ("scd1_contains_no_history", int(scd1_metrics["historical_rows"] or 0) == 0),
    ("scd2_one_current_row_per_key", scd2_metrics["current_rows"] == scd2_metrics["business_keys"]),
    ("scd2_retains_history", int(scd2_metrics["historical_rows"] or 0) > 0),
    ("business_key_coverage_matches", missing_from_scd1 == 0 and missing_from_scd2 == 0),
    ("scd1_quality_rules_pass", all(value == 0 for value in scd1_checks.values())),
    ("scd2_quality_rules_pass", all(value == 0 for value in scd2_checks.values())),
]
failed_checks = [name for name, passed in final_checks if not passed]
if run_validation and failed_checks:
    raise AssertionError(f"Final SCD comparison checks failed: {failed_checks}")

final_validation_df = spark.createDataFrame(
    [(name, "PASS" if passed else "FAIL") for name, passed in final_checks],
    ["validation", "status"],
)
display(final_validation_df)
print("✅ SCD Type 1 and Type 2 comparison completed successfully.")

## Evidence to capture

Save screenshots of:

1. the SCD1/SCD2 dimension metrics table;
2. both zero-violation quality-rule tables;
3. the changed-product side-by-side comparison;
4. the SCD2 multi-version history rows;
5. the final all-PASS validation table.

Suggested filenames: `07_scd_metrics.png`, `08_scd_history_comparison.png`, and `09_scd_comparison_validation.png`.

## Next notebook

Continue with **`lab04_08_schema_enforcement.ipynb`**. It will apply an explicit Delta schema, intentionally attempt incompatible writes, prove that enforcement rejects them, and compare fail-fast contract behavior with quarantine/rescue handling.